In [16]:
import easyocr
reader = easyocr.Reader(["en"], gpu=True)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [9]:


path = "../data/image/img_1.jpg"
result = reader.readtext(path, detail=0, paragraph=True)
print("\n".join(result))

c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


WGA Keep The JAR CLOSED TO PRESERVE FRESHNESS  INGREDIENTS : Foxnut (Makhana), Olive Oil; Rock Salt
PROPRIETARY FOODS (Under Categorv -15


In [17]:
import re

def extract_ingredients(ocr_text):
    t = " ".join(ocr_text) if isinstance(ocr_text, list) else ocr_text
    m = re.search(r"ingredients?\s*[:\-]?\s*(.*)", t, flags=re.I | re.S)
    if not m:
        return None                      # no ingredient list found -> ask user to retake
    body = m.group(1)
    # stop at the next section heading
    body = re.split(r"(nutrition|nutritional|allergen|contains|may contain|best before|"
                    r"manufactured|marketed|mfd|net wt|storage)", body, flags=re.I)[0]
    return body.strip(" .;:")

def split_ingredients(body):
    parts, depth, cur = [], 0, ""
    for ch in body:
        if ch in "([{": depth += 1
        elif ch in ")]}": depth = max(0, depth - 1)
        if ch in ",;" and depth == 0:
            parts.append(cur.strip()); cur = ""
        else:
            cur += ch
    if cur.strip(): parts.append(cur.strip())
    return parts

text = reader.readtext(path, detail=0, paragraph=True)
body = extract_ingredients(text)
print(body)
print(split_ingredients(body))

c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Foxnut (Makhana), Olive Oil; Rock Salt PROPRIETARY FOODS (Under Categorv -15
['Foxnut (Makhana)', 'Olive Oil', 'Rock Salt PROPRIETARY FOODS (Under Categorv -15']


In [18]:
import os, pandas as pd

folder = "../data/image"
rows = []
for f in sorted(os.listdir(folder)):
    if f.lower().endswith((".jpg", ".jpeg", ".png")):
        raw = reader.readtext(f"{folder}/{f}", detail=0, paragraph=True)
        body = extract_ingredients(raw)
        rows.append({
            "image": f,
            "found": body is not None,
            "ocr_ingredients": body,
            "items": split_ingredients(body) if body else [],
        })

res = pd.DataFrame(rows)
res.to_csv(f"{folder}/ocr_draft.csv", index=False)
print(res["found"].value_counts())
res

c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\U

found
True     13
False     7
Name: count, dtype: int64


,image,found,ocr_ingredients,items
0,img_1.jpg,True,"Foxnut (Makhana), Olive Oil; Rock Salt PROPRIE...","[Foxnut (Makhana), Olive Oil, Rock Salt PROPRI..."
1,img_10.jpg,True,"~SUGAR, COcOA SOLIDS COcOA BUTTER PERMITTED E...","[~SUGAR, COcOA SOLIDS COcOA BUTTER PERMITTED ..."
2,img_11.jpg,True,Relined Sugar Refined Wheat Flour (Maida Refin...,[Relined Sugar Refined Wheat Flour (Maida Refi...
3,img_12.jpg,True,"NoodLes: WHEAT FLCUP . Palm QiSALT, WHEAT GLuT...","[NoodLes: WHEAT FLCUP . Palm QiSALT, WHEAT GLu..."
4,img_13.jpg,True,SUGAR; cOcOA SOLIDS; COCOA BUTTER; ALMONDS(896...,"[SUGAR, cOcOA SOLIDS, COCOA BUTTER, ALMONDS(89..."
5,img_14.jpg,True,Sugar Syrup (Water 8 'Sugar) Mik Casein (Chen...,[Sugar Syrup (Water 8 'Sugar) Mik Casein (Che...
6,img_15.jpg,False,NaN,[]
7,img_16.jpg,True,"REFINED WHEAT FLOUR, CHICKPEA FLOUR, REFINED P...","[REFINED WHEAT FLOUR, CHICKPEA FLOUR, REFINED ..."
8,img_17.jpg,False,NaN,[]
9,img_18.jpg,True,Cereal Products (6698) (Rice Meal (43598 Cor M...,[Cereal Products (6698) (Rice Meal (43598 Cor ...


In [19]:
import re, difflib

STOP = r"(nutrition|nutritional|allergen|contains|may contain|best before|manufactured|marketed|" \
       r"mfd|net wt|storage|fssai|proprietary|permitted|customer care|packed by|consume)"

def extract_ingredients(ocr_text):
    t = " ".join(ocr_text) if isinstance(ocr_text, list) else str(ocr_text)
    words = t.split()
    start = None
    for i, w in enumerate(words):
        w_clean = re.sub(r"[^a-z]", "", w.lower())
        if len(w_clean) >= 7 and difflib.SequenceMatcher(None, w_clean, "ingredients").ratio() >= 0.75:
            start = i
            break
    if start is None:
        return None
    body = " ".join(words[start + 1:])
    body = re.sub(r"^[\s:\-]+", "", body)
    body = re.split(STOP, body, flags=re.I)[0]
    return body.strip(" .;:") or None

In [20]:
failed = res[~res["found"]]["image"].tolist()
for f in failed:
    raw = reader.readtext(f"{folder}/{f}", detail=0, paragraph=True)
    print(f, "->", " ".join(raw)[:300], "\n")

c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


img_15.jpg -> 4nnaa Cootederky) , SusAR, Lemon, Mlvit Rune 



c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


img_17.jpg -> (( We no Iohoutt 6 4ra JTteL-Tt OIeI Memn cllulnhkralor Jn  Dutepeteto  Aant][ cneaenAnd d 013n*d0Jitmidornu0 Inenedict Lol Kcunne ecir " Iih Oan L 



c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


img_19.jpg -> 1 8 1 6 M fssal 10012022000908 JEstewwio Wlati) d o 405,Ml Grta Ua FEdE 



... (4 outputs left out)

c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


img_6.jpg -> Dark ChOGREDIENTS Chocolate Roasted [ (1091, Peanuts (8334 Allergen Sugar; Salt; Stabiizer (UNS4711] Info : Contains Peanurts and Soya: Peanut Based Spread with Dark Chocolate (Proprietary  Food-4.2.245-Spreadj 



c:\Users\HP\Desktop\food-safety-advisor\.venv-1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


img_8.jpg -> SARCTNC PFCOHNBFPCD A AnE EOBLPFOR! OHISA Rosegition Parle G Gluco BISCUITS WGREOIENTS : REFINED WHEAT FLOUR SUGAR, REFINED PALM OIL , INVERT BGMAIDHRU7S Sugar, NTRICACIDL; (ODISED SALT, RAISING AGENIS 15o30), Soorh1; Ilk SOLIDS , ARTIFICIAL (VANILLA) FLAVOURING SUBSTANCES ' EMULSIFIER @F VEGETABIE  



In [15]:
import cv2
img = cv2.imread(f"{folder}/img_13.jpg")
big = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
raw = reader.readtext(big, detail=0, paragraph=True)
print(" ".join(raw)[:400])

K U 2 Rladt i0Usi Iavan%d ,Ava HI Maa (aaoan WruIc Q Kquam Jotl (ool AD Duar @eshertoune WLu Dobuctnina Iad


In [21]:
import re, difflib, warnings
warnings.filterwarnings("ignore")        # hides the harmless pin_memory warnings

STOP = (r"(nutrition|nutritional|allergen|contains|may contain|best before|manufactured|marketed|"
        r"mfd|net wt|storage|fssai|proprietary|permitted|customer care|packed by|consume)")

def find_header_end(text, threshold=0.7):
    low = text.lower()
    best_ratio, best_end = 0, None
    for L in (9, 10, 11, 12):
        for i in range(len(low) - L + 1):
            r = difflib.SequenceMatcher(None, low[i:i+L], "ingredients").ratio()
            if r > best_ratio:
                best_ratio, best_end = r, i + L
    return (best_end, best_ratio) if best_ratio >= threshold else (None, best_ratio)

def extract_ingredients(ocr_text):
    t = " ".join(ocr_text) if isinstance(ocr_text, list) else str(ocr_text)
    end, ratio = find_header_end(t)
    if end is None:
        return None
    body = re.sub(r"^[\s:;\-\]\)]+", "", t[end:])
    body = re.split(STOP, body, flags=re.I)[0].strip(" .;:")
    return body if len(re.findall(r"[,;]", body)) >= 2 else None

In [23]:
from collections import Counter
df = pd.read_csv("../data/processed/food_safety_clean (1).csv")
df["items"] = df["ingredients_clean"].map(split_ingredients)
counter = Counter(item.lower() for items in df["items"] for item in items)
print(len(counter), "distinct ingredient strings")
counter.most_common(40)

5130 distinct ingredient strings


[('sugar', 467),
 ('water', 210),
 ('milk solids', 165),
 ('salt', 154),
 ('iodised salt', 105),
 ('refined wheat flour (maida)', 95),
 ('iodized salt', 81),
 ('lodised salt', 69),
 ('garlic', 65),
 ('yeast', 60),
 ('sunflower oil', 57),
 ('liquid glucose', 57),
 ('milk', 53),
 ('lodized salt', 48),
 ('cocoa solids', 47),
 ('ginger', 47),
 ('maltodextrin', 45),
 ('refined palm oil', 43),
 ('wheat gluten', 41),
 ('rice flour', 38),
 ('starch', 36),
 ('onion', 33),
 ('invert sugar syrup', 33),
 ('cocoa butter', 32),
 ('cumin', 32),
 ('palm oil', 31),
 ('black pepper', 30),
 ('dextrose', 29),
 ('carbonated water', 28),
 ('butter', 27),
 ('peanuts', 27),
 ('wheat', 26),
 ('acidity regulator (ins 330)', 25),
 ('nature identical flavouring substances', 25),
 ('cocoa powder', 24),
 ('refined palmolein oil', 24),
 ('minerals', 22),
 ('coriander', 22),
 ('turmeric', 22),
 ('edible common salt', 22)]

In [24]:
total = sum(counter.values())
for n in (50, 100, 300, 1000):
    covered = sum(c for _, c in counter.most_common(n))
    print(f"Top {n} strings cover {covered/total:.1%} of all ingredient mentions")

Top 50 strings cover 27.0% of all ingredient mentions
Top 100 strings cover 34.0% of all ingredient mentions
Top 300 strings cover 45.2% of all ingredient mentions
Top 1000 strings cover 58.6% of all ingredient mentions


In [26]:
import sys; sys.path.append("..")
from src.normalize import canonical

df["canon"] = df["items"].map(lambda items: [canonical(i) for i in items])
c2 = Counter(x for items in df["canon"] for x in items)
print(len(counter), "->", len(c2), "distinct strings")
c2.most_common(40)

5130 -> 4130 distinct strings


[('sugar', 498),
 ('salt', 445),
 ('milk solids', 219),
 ('water', 212),
 ('refined wheat flour', 149),
 ('palm oil', 88),
 ('cocoa solids', 74),
 ('garlic', 74),
 ('glucose syrup', 64),
 ('yeast', 62),
 ('sunflower oil', 58),
 ('milk', 54),
 ('ginger', 54),
 ('peanuts', 53),
 ('rice flour', 51),
 ('iodized salt', 49),
 ('maltodextrin', 47),
 ('wheat gluten', 44),
 ('onion', 41),
 ('cocoa powder', 38),
 ('invert sugar', 38),
 ('starch', 36),
 ('wheat', 35),
 ('cocoa butter', 33),
 ('butter', 33),
 ('cumin', 33),
 ('black pepper', 33),
 ('acidity regulator ins 330', 32),
 ('tomato paste', 31),
 ('potato', 30),
 ('dextrose', 29),
 ('nature identical flavouring substances', 28),
 ('carbonated water', 28),
 ('coriander', 26),
 ('refined palm oil oil', 26),
 ('dates', 26),
 ('minerals', 25),
 ('vitamins', 25),
 ('turmeric', 25),
 ('jaggery', 24)]

In [27]:
from rapidfuzz import process, fuzz

known = [name for name, n in c2.most_common(500)]         # common, trusted names

def fuzzy_fix(name, cutoff=88):
    if name in known: return name
    match = process.extractOne(name, known, scorer=fuzz.ratio, score_cutoff=cutoff)
    return match[0] if match else name

In [28]:
pd.DataFrame(c2.most_common(300), columns=["ingredient", "count"]).to_csv(
    "../data/processed/ingredient_vocab_top300.csv", index=False)

In [29]:
total = sum(c2.values())
for n in (50, 100, 300, 500, 1000):
    covered = sum(c for _, c in c2.most_common(n))
    print(f"Top {n} names cover {covered/total:.1%} of mentions")

singletons = [k for k, v in c2.items() if v == 1]
print(len(singletons), "names appear only once",
      f"({len(singletons)/total:.1%} of mentions)")

import random
random.seed(1)
for s in random.sample(singletons, 40):
    print(s)

Top 50 names cover 32.0% of mentions
Top 100 names cover 40.4% of mentions
Top 300 names cover 53.7% of mentions
Top 500 names cover 60.1% of mentions
Top 1000 names cover 68.6% of mentions
3274 names appear only once (32.8% of mentions)
mb enzymepro
pasteurised toned milk milk solids and active lactic culture
artificial mango flavour and colour 160a iii may contain milk and soy
raising agents 503 500 i
antioxidant ins 386
thickeners 1422 415
sequestrant ins 451 i
dried citrus pulp
iodised salt acidity regulator citric acid ins 330
honey allergen caution contains tree nuts this product has been packed in an integrated nuts dried fruits processing unit and may contain an occasional trace of other nuts or dried fruits
organic curry leaves
permitted class ii preservatives ins 211 permitted natural food colour ins 150d and added flavours natural nature identical and artificial flavouring substances cranberry
organic fruit and vegetable juice for color
icing sugar
gram dal
nuts seed mix alm

In [33]:



from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))
from src.parser import parse_ingredient

tests = [
    "raising agents 503 500 i",
    "thickeners 1422 415",
    "iodised salt acidity regulator citric acid ins 330",
    "honey allergen caution contains tree nuts this product has been packed in an integrated nuts",
    "permitted class ii preservatives ins 211 permitted natural food colour ins 150d and added flavours natural nature identical and artificial flavouring substances cranberry",
]
for t in tests:
    print(t[:50], "->", parse_ingredient(t))

raising agents 503 500 i -> ['raising agent', 'ins 503', 'ins 500']
thickeners 1422 415 -> ['thickener', 'ins 1422', 'ins 415']
iodised salt acidity regulator citric acid ins 330 -> ['salt', 'acidity regulator', 'ins 330', 'citric acid']
honey allergen caution contains tree nuts this pro -> ['honey']
permitted class ii preservatives ins 211 permitted -> ['preservative', 'ins 211', 'colour', 'ins 150d', 'added flavour']


In [39]:
import sys, traceback, pandas as pd
from collections import Counter
sys.path.append("..")

import importlib, src.parser, src.normalize
importlib.reload(src.normalize)
importlib.reload(src.parser)                 # pick up any edits you saved
from src.parser import split_ingredients, parse_ingredient

df = pd.read_csv("../data/processed/food_safety_clean (1).csv")
df["items"] = df["ingredients_clean"].fillna("").map(split_ingredients)

canon3 = []
for i, items in df["items"].items():
    try:
        canon3.append([c for x in items for c in parse_ingredient(x)])
    except Exception:
        print("FAILED on row", i, "->", items)
        traceback.print_exc()
        break
else:
    df["canon3"] = canon3
    c3 = Counter(x for L in df["canon3"] for x in L)
    print(len(c3), "distinct names")
    print(c3.most_common(40))

3920 distinct names
[('salt', 654), ('sugar', 587), ('added flavour', 559), ('emulsifier', 378), ('acidity regulator', 350), ('milk solids', 265), ('water', 234), ('colour', 223), ('ins 330', 191), ('stabilizer', 179), ('preservative', 170), ('refined wheat flour', 155), ('ins 322', 145), ('ins 500', 143), ('palm oil', 133), ('raising agent', 130), ('ins 471', 126), ('cocoa solids', 123), ('antioxidant', 123), ('garlic', 96), ('ginger', 89), ('maltodextrin', 88), ('flavour enhancer', 86), ('black pepper', 83), ('glucose syrup', 73), ('cumin', 73), ('yeast', 72), ('milk', 70), ('sunflower oil', 69), ('anticaking agent', 69), ('ins 503', 66), ('thickener', 65), ('rice flour', 60), ('ins 415', 58), ('ins 150d', 58), ('ins 412', 57), ('ins 211', 57), ('ins 260', 56), ('wheat gluten', 55), ('wheat', 55)]


In [40]:
total = sum(c3.values())
for n in (50, 100, 300, 500, 1000):
    print(f"Top {n}: {sum(c for _, c in c3.most_common(n))/total:.1%} of mentions")

print(sum(1 for v in c3.values() if v == 1), "names appear only once")

odd = [k for k in c3 if k.isdigit() or len(k) < 3 or len(k) > 60]
print(len(odd), odd[:30])

print((df["canon3"].str.len() == 0).sum(), "products with an empty ingredient list")

Top 50: 41.2% of mentions
Top 100: 51.4% of mentions
Top 300: 67.5% of mentions
Top 500: 73.9% of mentions
Top 1000: 81.5% of mentions
2774 names appear only once
156 ['edible vegetable oil luntains one or more followings palm oil', 'diacetyltartaric and fatty protein acid esters of glycerol from sunflower oil palm oil', 'diacetyltartaric and fatty acid esters of glycerol from refined palm oil and sunflower oil', 'diacety tartaric and fatty acid esters of glycerol from refined sunflower oil and palm oil', 'diacetyl tartaric acid ester of mono and diglycerides from sunflower palm oil', '551', '500', 'refined edible palm oil oil and iodised salt tomato chilli seasoning sachet net quantity 4g dehydrated vegetables tomato powder', 'iodised salt spices condiments salt substitute potassium chloride', '631', 'as', 'diacetyltartaric and fatty acid esters of glycerol from refined sunflower and palm oil', 'a', 'ns', 'dough conditioner 223 and minerals numbers in brackets as per international num

In [41]:
for name, n in c3.most_common(40):
    print(n, name)

654 salt
587 sugar
559 added flavour
378 emulsifier
350 acidity regulator
265 milk solids
234 water
223 colour
191 ins 330
179 stabilizer
170 preservative
155 refined wheat flour
145 ins 322
143 ins 500
133 palm oil
130 raising agent
126 ins 471
123 cocoa solids
123 antioxidant
96 garlic
89 ginger
88 maltodextrin
86 flavour enhancer
83 black pepper
73 glucose syrup
73 cumin
72 yeast
70 milk
69 sunflower oil
69 anticaking agent
66 ins 503
65 thickener
60 rice flour
58 ins 415
58 ins 150d
57 ins 412
57 ins 211
56 ins 260
55 wheat gluten
55 wheat


In [42]:
pd.DataFrame(c3.most_common(300), columns=["ingredient", "count"]).to_csv(
    "../data/processed/ingredient_vocab_top300.csv", index=False)